In [ ]:
# !WANDB_PROJECT="cfe_finetuning" python3 -m wandb sweep sweep.yaml

In [4]:
import pandas as pd
import json

In [22]:
import pandas as pd
import json

with open("finetuning_eval/pythia-31m_tulu-v2-sft-mixture_train-merged/metrics.json", "r") as f:
    data = json.load(f)


scores_list = []
for entry in data.get("all_primary_scores", []):
    alias, score = entry.rsplit(": ", 1)
    scores_list.append({"alias": alias, "primary_score": float(score)})
df_primary_scores = pd.DataFrame(scores_list)


tasks_list = []
for task in data.get("tasks", []):

    entry = {"alias": task["alias"], "num_instances": task.get("num_instances")}
    entry.update(task.get("metrics", {}))
    tasks_list.append(entry)
df_tasks = pd.DataFrame(tasks_list)

df = pd.merge(df_primary_scores, df_tasks, on="alias", how="left")
df


,alias,primary_score_x,num_instances,primary_score_micro,primary_score_macro,logits_per_char_corr_micro,logits_per_char_corr_macro,sum_logits_corr_micro,sum_logits_corr_macro,bits_per_byte_corr_micro,...,acc_per_char,acc_per_byte,sum_logits_corr,logits_per_token_corr,logits_per_char_corr,bits_per_byte_corr,exact_match_simple,exact_match,f1,recall
0,agi_eval_english:1shot::olmes,0.232114,2646,0.223356,0.232114,-1.908329,-1.898482,-3.816658,-3.796964,2.753137,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,bbh:cot-v1::olmes,0.110134,6511,0.104131,0.110134,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,mmlu_pro:mc::none,0.112866,12032,0.112866,0.113261,-2.068762,-2.023758,-4.137524,-4.047515,2.984593,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,agi_eval_lsat-ar:1shot::olmes,0.208696,230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.208696,0.208696,-3.515195,-3.515195,-1.757597,2.535677,NaN,NaN,NaN,NaN
4,agi_eval_lsat-lr:1shot::olmes,0.200000,510,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.200000,0.200000,-3.470496,-3.470496,-1.735248,2.503434,NaN,NaN,NaN,NaN
5,agi_eval_lsat-rc:1shot::olmes,0.189591,269,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.189591,0.189591,-3.213456,-3.213456,-1.606728,2.318018,NaN,NaN,NaN,NaN
6,agi_eval_logiqa-en:1shot::olmes,0.202765,651,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.202765,0.202765,-4.147406,-4.147406,-2.073703,2.991721,NaN,NaN,NaN,NaN
7,agi_eval_sat-math:1shot::olmes,0.240909,220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.240909,0.240909,-4.780677,-4.780677,-2.390338,3.448529,NaN,NaN,NaN,NaN
8,agi_eval_sat-en:1shot::olmes,0.310680,206,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.310680,0.310680,-3.832254,-3.832254,-1.916127,2.764387,NaN,NaN,NaN,NaN
9,agi_eval_aqua-rat:1shot::olmes,0.255906,254,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.255906,0.255906,-2.971387,-2.971387,-1.485694,2.143403,NaN,NaN,NaN,NaN


In [24]:
df["benchmark"] = df["alias"].apply(lambda x: x.split("_")[0])

In [29]:
df.groupby("benchmark")[["primary_score_x", ]].mean()

,primary_score_x
benchmark,
agi,0.232114
bbh,0.110134
bbh:cot-v1::olmes,0.110134
gsm8k::olmes,0.005307
mmlu,0.113235
triviaqa::olmes,0.034136


In [2]:
# pip install pygsheets

In [3]:
# pip install lm_eval

In [4]:
!nvidia-smi

Tue Sep  9 15:07:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.163.01             Driver Version: 550.163.01     CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla V100-SXM2-16GB           On  |   00000000:85:00.0 Off |                    0 |
| N/A   31C    P0             42W /  300W |       1MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# !python3 ./merge_lora_to_hf.py \
#   --base_model EleutherAI/pythia-31m \
#   --adapter_dir /srv/home/users/loriss21cs/cfe/models/pythia-31m_tulu-v2-sft-mixture_train \
#   --output_dir /srv/home/users/loriss21cs/cfe/models/pythia-31m_tulu-v2-sft-mixture_train-merged


In [3]:
import multiprocessing as mp
mp.set_start_method('spawn', force=True)


In [ ]:
!

2025-09-09 15:11:15,499 [WARNING] No config found for model:/srv/home/users/loriss21cs/cfe/models/pythia-31m_tulu-v2-sft-mixture_train-merged, using raw model
2025-09-09 15:11:15,501 [INFO] Running eval locally on 99 tasks!
2025-09-09 15:11:15,501 [INFO] Command: python3 -m oe_eval.run_eval --task '{"task_name": "gsm8k", "split": "test", "primary_metric": "exact_match", "use_chat_format": true, "num_shots": 8, "fewshot_source": "STD:GSM8k", "random_subsample_seed": 42, "chat_overrides": {"context_kwargs": {"fewshot_as_multiturn": true}}, "metadata": {"regimes": ["Tulu"], "alias": "gsm8k::tulu"}}' '{"task_name": "drop", "split": "validation", "primary_metric": "f1", "num_shots": 3, "metadata": {"regimes": ["Llama-3"], "alias": "drop::llama3"}}' '{"task_name": "minerva_math_algebra", "split": "test", "use_chat_format": true, "num_shots": 4, "chat_overrides": {"context_kwargs": {"fewshot_as_multiturn": true}}, "generation_kwargs": {"max_gen_toks": 1024, "temperature": 0.0, "do_sample": fa

In [7]:
# import subprocess
# import os

# # Change working directory to olmes
# olmes_dir = os.path.join("olmes")
# os.chdir(olmes_dir)

# cmd = [
#     "CUDA_VISIBLE_DEVICES=0 python3", "-m", "oe_eval.launch",
#     "--model","/srv/home/users/loriss21cs/cfe/models/pythia-31m_tulu-v2-sft-mixture_train-merged",
#     "--task", "tulu_3_dev",
#     "--output-dir", "./finetuning_eval/"
  
# ]

# subprocess.run(cmd, check=True)
